# MobileNetV2 — Inverted Residuals and Linear Bottlenecks (TensorFlow / Keras)
**Paper:** MobileNetV2: Inverted Residuals and Linear Bottlenecks (Sandler et al., CVPR 2018)

| Property | Value |
|----------|-------|
| Framework | TensorFlow / Keras |
| Block type | Inverted Residual (Expand→DWConv→Linear Project) |
| Inverted Residual Blocks | 17 |
| Parameters | ~3.5M |
| Top-1 (ImageNet) | ~71.3% |
| Input size | 224×224 |
| Preprocessing | mobilenet_v2.preprocess_input → x/127.5 − 1.0 → [−1, 1] |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {bool(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# Cell 2 — MobileNetV2 Architecture (from scratch)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def _make_divisible(v, divisor=8, min_value=None):
    """Ensure channel count is divisible by divisor (needed for width multiplier)."""
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def _inverted_residual(x, filters, stride, expand_ratio, block_id):
    """
    MobileNetV2 Inverted Residual Block (Linear Bottleneck).

    Structure: Expand -> Depthwise -> Project (linear, no activation).
    Residual added only when stride=1 AND input channels == output channels.

    Key insight: the bottleneck is 'inverted' vs ResNet:
      ResNet   : wide -> narrow -> wide  (compress then expand)
      V2       : narrow -> wide -> narrow (expand then project)

    The final projection has NO activation (linear bottleneck) to preserve
    the information manifold before the residual addition.

    block_id=0: special case 'expanded_conv' (t=1, no expand phase, no residual)
    block_id>0: 'block_{block_id}_expand / depthwise / project'
    """
    in_channels  = x.shape[-1]
    prefix       = 'expanded_conv' if block_id == 0 else f'block_{block_id}'
    shortcut     = x

    # ── Expand ──
    if expand_ratio != 1:
        exp_ch = _make_divisible(in_channels * expand_ratio)
        x = layers.Conv2D(exp_ch, 1, padding='same', use_bias=False,
                          name=f'{prefix}_expand')(x)
        x = layers.BatchNormalization(epsilon=1e-3, momentum=0.999,
                                      name=f'{prefix}_expand_BN')(x)
        x = layers.ReLU(6., name=f'{prefix}_expand_relu')(x)

    # ── Depthwise ──
    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False,
                                name=f'{prefix}_depthwise')(x)
    x = layers.BatchNormalization(epsilon=1e-3, momentum=0.999,
                                  name=f'{prefix}_depthwise_BN')(x)
    x = layers.ReLU(6., name=f'{prefix}_depthwise_relu')(x)

    # ── Project (linear — no activation) ──
    x = layers.Conv2D(filters, 1, padding='same', use_bias=False,
                      name=f'{prefix}_project')(x)
    x = layers.BatchNormalization(epsilon=1e-3, momentum=0.999,
                                  name=f'{prefix}_project_BN')(x)

    # ── Residual: only when stride=1 and channels match ──
    if stride == 1 and in_channels == filters:
        x = layers.Add(name=f'{prefix}_add')([shortcut, x])

    return x


def build_mobilenetv2(num_classes=1000, input_shape=(224, 224, 3), alpha=1.0):
    """
    MobileNetV2 — Inverted Residuals and Linear Bottlenecks.
    Paper: Sandler et al., CVPR 2018.

    Key improvements over MobileNetV1:
      1. Inverted Residuals: expand channels (6x) -> DWConv -> project back
      2. Linear Bottleneck: no activation on the projection layer
      3. Residual connections: skip connection when stride=1 and dims match
      4. Better accuracy at same or fewer parameters vs V1

    Inverted Residual Block config: (t=expand_ratio, c=channels, n=repeats, s=stride)

    t   c    n  s
    1   16   1  1   <- expanded_conv (no expand, t=1)
    6   24   2  2
    6   32   3  2
    6   64   4  2
    6   96   3  1
    6  160   3  2
    6  320   1  1
    Then: Conv1x1(1280) + BN + ReLU6  -> GAP -> Dense
    """
    # Inverted residual configs: (expand_ratio, channels, num_repeats, stride)
    configs = [
        (1,  16, 1, 1),
        (6,  24, 2, 2),
        (6,  32, 3, 2),
        (6,  64, 4, 2),
        (6,  96, 3, 1),
        (6, 160, 3, 2),
        (6, 320, 1, 1),
    ]

    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(_make_divisible(32 * alpha), 3, strides=2,
                      padding='same', use_bias=False, name='Conv1')(inputs)
    x = layers.BatchNormalization(epsilon=1e-3, momentum=0.999,
                                  name='bn_Conv1')(x)
    x = layers.ReLU(6., name='Conv1_relu')(x)

    block_id = 0
    for t, c, n, s in configs:
        out_ch = _make_divisible(c * alpha)
        for i in range(n):
            stride = s if i == 0 else 1
            x = _inverted_residual(x, out_ch, stride, t, block_id)
            block_id += 1

    # Head conv: expand to 1280 channels before GAP
    last_ch = max(_make_divisible(1280 * alpha), 1280)
    x = layers.Conv2D(last_ch, 1, padding='same', use_bias=False,
                      name='Conv_1')(x)
    x = layers.BatchNormalization(epsilon=1e-3, momentum=0.999,
                                  name='Conv_1_bn')(x)
    x = layers.ReLU(6., name='out_relu')(x)

    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax',
                           name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='mobilenetv2')


In [ ]:
NUM_CLASSES = 10
INPUT_SHAPE = (224, 224, 3)

model = build_mobilenetv2(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)
model.summary(line_length=120)

In [ ]:
dummy  = tf.random.normal((2, 224, 224, 3))
output = model(dummy, training=False)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
non_trainable    = total_params - trainable_params
print(f"{'='*48}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable         : {non_trainable:,}")
print(f"{'='*48}")

In [ ]:
print(f"{'Layer':<55} {'Output Shape':<30} {'Params':>10}")
print('-' * 98)
total = 0
for layer in model.layers:
    params = layer.count_params()
    total += params
    try:
        out_shape = str(layer.output_shape)
    except Exception:
        out_shape = 'multiple'
    print(f"{layer.name:<55} {out_shape:<30} {params:>10,}")
print('-' * 98)
print(f"{'TOTAL':<86} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, zoom_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{DATA_DIR}/train', target_size=(224, 224),
    batch_size=BATCH_SIZE, class_mode='categorical',
)
val_gen = val_datagen.flow_from_directory(
    f'{DATA_DIR}/val', target_size=(224, 224),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False,
)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {train_gen.samples}')
print(f'Val size   : {val_gen.samples}')

In [ ]:
EPOCHS = 30

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
callbacks = [
    keras.callbacks.ModelCheckpoint(
        'mobilenetv2_best.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.1, patience=5, min_lr=1e-7, verbose=1,
    ),
]
history = model.fit(
    train_gen, epochs=EPOCHS, validation_data=val_gen,
    callbacks=callbacks, verbose=1,
)
print('Best model saved: mobilenetv2_best.keras')

---
## Training Curves

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MobileNetV2 — Training Curves', fontsize=14, fontweight='bold')
axes[0].plot(epochs_range, history.history['loss'],         'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'],     'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
def predict_image(image_path, top_k=5):
    img    = keras.utils.load_img(image_path, target_size=(224, 224))
    arr    = keras.utils.img_to_array(img) / 255.0
    tensor = tf.expand_dims(arr, 0)
    probs  = model(tensor, training=False)[0].numpy()
    top_idx = probs.argsort()[::-1][:top_k]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input')
    axes[1].barh([CLASS_NAMES[i] for i in top_idx][::-1],
                 [probs[i]*100 for i in top_idx][::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({probs[top_idx[0]]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
val_gen.reset()
y_pred     = model.predict(val_gen, steps=len(val_gen), verbose=1)
y_true     = val_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print(f'Predictions shape : {y_pred.shape}')
print(f'Labels shape      : {y_true_bin.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {auc(fpr,tpr):.3f})')
macro_auc = roc_auc_score(y_true_bin, y_pred, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'MobileNetV2 — ROC AUC  (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')